# House Price Prediction - Data Cleaning and Preprocessing

This notebook prepares the housing data for a later modeling step. It cleans only clearly invalid records, separates the target from features, creates a reproducible train/test split, and fits imputers and one-hot encoding on the training data only.

**No model is trained and the original CSV is never changed.**

## 1. Load and inspect the original data

The EDA notebook and the CSV show that the source file is `Housing.csv`, its target column is `price`, and it contains 12 potential input features. The relative path below is written from the `notebooks` folder.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

data_path = Path("../dataset/Housing.csv")
raw_houses = pd.read_csv(data_path)

print(f"Loaded: {data_path.name}")
print("Shape:", raw_houses.shape)
print("Columns:", raw_houses.columns.tolist())
raw_houses.head()

Loaded: Housing.csv
Shape: (545, 13)
Columns: ['price', 'area', 'bedrooms', 'bathrooms', 'stories', 'mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'parking', 'prefarea', 'furnishingstatus']


,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


## 2. Clean only confirmed data problems

- Exact duplicate rows are redundant, so remove them if any are present.
- Rows without a house price cannot be used as labeled examples, so remove them if present.
- A price, area, bedroom, bathroom, or story count must be positive. Parking may be **zero** (no parking spaces), but not negative. Remove rows only when these observed values violate those rules.
- Strip surrounding spaces and lowercase category text to make values consistent. Unexpected category labels are reported as an error instead of being silently changed or discarded.
- Feature missing values are left for the training-fitted imputers in the next step.

In [2]:
target_column = "price"
if target_column not in raw_houses.columns:
    raise KeyError(f"Target column {target_column!r} was not found. Review the dataset columns.")

houses = raw_houses.copy()

duplicate_rows_removed = int(houses.duplicated().sum())
houses = houses.drop_duplicates().copy()

missing_target_rows_removed = int(houses[target_column].isna().sum())
houses = houses.dropna(subset=[target_column]).copy()

categorical_columns = houses.select_dtypes(include=["object", "category", "string"]).columns.tolist()
normalized_category_values = 0
for column in categorical_columns:
    original_values = houses[column].astype("string")
    cleaned_values = original_values.str.strip().str.lower()
    normalized_category_values += int((original_values.notna() & original_values.ne(cleaned_values)).sum())
    houses[column] = cleaned_values

expected_categories = {
    "mainroad": {"yes", "no"},
    "guestroom": {"yes", "no"},
    "basement": {"yes", "no"},
    "hotwaterheating": {"yes", "no"},
    "airconditioning": {"yes", "no"},
    "prefarea": {"yes", "no"},
    "furnishingstatus": {"furnished", "semi-furnished", "unfurnished"},
}
unexpected_categories = {}
for column, allowed_values in expected_categories.items():
    if column in houses.columns:
        observed_values = set(houses[column].dropna().unique())
        unexpected_values = observed_values - allowed_values
        if unexpected_values:
            unexpected_categories[column] = sorted(unexpected_values)
if unexpected_categories:
    raise ValueError(f"Review unexpected categorical values before continuing: {unexpected_categories}")

positive_columns = [
    column for column in ["price", "area", "bedrooms", "bathrooms", "stories"]
    if column in houses.columns
]
invalid_numeric_rows = pd.Series(False, index=houses.index)
for column in positive_columns:
    invalid_numeric_rows |= houses[column].notna() & houses[column].le(0)
if "parking" in houses.columns:
    invalid_numeric_rows |= houses["parking"].notna() & houses["parking"].lt(0)

invalid_numeric_rows_removed = int(invalid_numeric_rows.sum())
houses = houses.loc[~invalid_numeric_rows].copy()

print("Duplicate rows removed:", duplicate_rows_removed)
print("Rows removed because price was missing:", missing_target_rows_removed)
print("Rows removed for invalid numeric values:", invalid_numeric_rows_removed)
print("Category values normalized:", normalized_category_values)
print("Cleaned data shape:", houses.shape)

Duplicate rows removed: 0
Rows removed because price was missing: 0
Rows removed for invalid numeric values: 0
Category values normalized: 0
Cleaned data shape: (545, 13)


## 3. Separate features and target, then split the data

`price` is the target (`y`); every other original column is retained as an input feature (`X`). The test set is held out before fitting any imputer or encoder. A fixed `random_state` makes the 80/20 split repeatable.

In [3]:
X = houses.drop(columns=[target_column]).copy()
y = houses[target_column].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

print("Target column:", target_column)
print("Feature columns retained:", X.columns.tolist())
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Target column: price
Feature columns retained: ['area', 'bedrooms', 'bathrooms', 'stories', 'mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'parking', 'prefarea', 'furnishingstatus']
Training rows: 436
Testing rows: 109


## 4. Build the preprocessing pipeline

Numeric feature gaps are filled with the **training median**. Categorical gaps are filled with the **most frequent training value**, then categories are one-hot encoded. `ColumnTransformer` applies the correct steps to each column group, and `Pipeline` keeps them together. `handle_unknown="ignore"` lets the encoder safely process a category found in test data but not in training data.

In [4]:
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category", "string"]).columns.tolist()
unassigned_features = set(X_train.columns) - set(numeric_features) - set(categorical_features)
if unassigned_features:
    raise TypeError(f"Review feature columns with unsupported data types: {sorted(unassigned_features)}")

numeric_pipeline = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="median"))]
)
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("one_hot_encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ],
    sparse_threshold=0.0,
)
preprocessing_pipeline = Pipeline(
    steps=[("preprocessor", preprocessor)]
)

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

Numeric features: ['area', 'bedrooms', 'bathrooms', 'stories', 'parking']
Categorical features: ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea', 'furnishingstatus']


## 5. Fit on training data and transform both sets

`fit_transform` learns imputation values and category encodings from `X_train` only. The test data is transformed with those already-fitted steps, so information from the test set cannot influence preprocessing.

In [5]:
X_train_processed = preprocessing_pipeline.fit_transform(X_train)
X_test_processed = preprocessing_pipeline.transform(X_test)

processed_feature_names = preprocessing_pipeline.named_steps["preprocessor"].get_feature_names_out()
train_features_have_missing_values = bool(np.isnan(X_train_processed).any())
test_features_have_missing_values = bool(np.isnan(X_test_processed).any())

assert X_train_processed.shape[1] == X_test_processed.shape[1]
assert not train_features_have_missing_values
assert not test_features_have_missing_values

print("Processed feature names:", processed_feature_names.tolist())
print("Processed training features contain missing values:", train_features_have_missing_values)
print("Processed testing features contain missing values:", test_features_have_missing_values)

Processed feature names: ['numeric__area', 'numeric__bedrooms', 'numeric__bathrooms', 'numeric__stories', 'numeric__parking', 'categorical__mainroad_no', 'categorical__mainroad_yes', 'categorical__guestroom_no', 'categorical__guestroom_yes', 'categorical__basement_no', 'categorical__basement_yes', 'categorical__hotwaterheating_no', 'categorical__hotwaterheating_yes', 'categorical__airconditioning_no', 'categorical__airconditioning_yes', 'categorical__prefarea_no', 'categorical__prefarea_yes', 'categorical__furnishingstatus_furnished', 'categorical__furnishingstatus_semi-furnished', 'categorical__furnishingstatus_unfurnished']
Processed training features contain missing values: False
Processed testing features contain missing values: False


## 6. Review the prepared data

The processed feature shape below combines the transformed training and testing rows; the transformations were still learned from training data only. The target summary describes the available house prices. The target itself is not imputed or encoded.

In [6]:
processed_feature_shape = (
    X_train_processed.shape[0] + X_test_processed.shape[0],
    X_train_processed.shape[1],
)

print("Processed feature shape (all rows):", processed_feature_shape)
print("Training feature shape:", X_train_processed.shape)
print("Training target shape:", y_train.shape)
print("Testing feature shape:", X_test_processed.shape)
print("Testing target shape:", y_test.shape)
print("\nTarget variable summary:")
display(y.describe())
print("Missing values handled in processed features:", not (train_features_have_missing_values or test_features_have_missing_values))

Processed feature shape (all rows): (545, 20)
Training feature shape: (436, 20)
Training target shape: (436,)
Testing feature shape: (109, 20)
Testing target shape: (109,)

Target variable summary:


count    5.450000e+02
mean     4.766729e+06
std      1.870440e+06
min      1.750000e+06
25%      3.430000e+06
50%      4.340000e+06
75%      5.740000e+06
max      1.330000e+07
Name: price, dtype: float64

Missing values handled in processed features: True


## 7. Preprocessing summary

- The original CSV was loaded but not modified.
- All original input features were retained; categorical features were expanded into one-hot columns.
- Duplicate rows, rows without a target, and rows violating basic numeric rules are removed only if they exist. A zero parking count remains valid.
- Remaining feature missing values are imputed by a pipeline fitted on training data only.
- The data is split 80/20 with `random_state=42`.
- No Linear Regression model has been trained.